# 01_ingesta_datos

## Objetivo
Este notebook realiza la ingesta inicial de los datasets del proyecto, verifica su estructura, revisa la consistencia de los nombres de los departamentos y deja preparada la base para construir el dataset maestro.

## Alcance de este notebook
- Cargar los archivos fuente desde la carpeta `data/raw/`
- Inspeccionar columnas, tipos de dato y cantidad de registros
- Estandarizar nombres de columnas clave
- Revisar la consistencia de la columna `departamento`
- Identificar diferencias entre departamentos presentes en cada fuente

## Requisitos previos
- Haber creado la estructura del proyecto
- Tener los archivos de datos dentro de `data/raw/`
- Estar usando el kernel del entorno virtual del proyecto


## 1. Importación de librerías

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Definición de rutas del proyecto



In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_RAW:', DATA_RAW)
print('DATA_PROCESSED:', DATA_PROCESSED)

PROJECT_ROOT: C:\Users\guill\microAI
DATA_RAW: C:\Users\guill\microAI\data\raw
DATA_PROCESSED: C:\Users\guill\microAI\data\processed


## 3. Verificación de archivos esperados

En esta sección se define el nombre esperado de cada archivo fuente. 

In [3]:
FILES = {
    'pobreza': DATA_RAW / 'pobreza_2024.xlsx',
    'microcredito': DATA_RAW / 'acceso_microcredito_2024.xlsx',
    'productos_financieros': DATA_RAW / 'acceso_productos_financieros_2024.xlsx',
    'atm': DATA_RAW / 'atm_x_10000_adultos_2024.xlsx',
    'internet': DATA_RAW / 'internet_hogares_2024.xlsx',
}

for nombre, ruta in FILES.items():
    print(f'{nombre:25s} -> {ruta.name:40s} | existe: {ruta.exists()}')

pobreza                   -> pobreza_2024.xlsx                        | existe: True
microcredito              -> acceso_microcredito_2024.xlsx            | existe: True
productos_financieros     -> acceso_productos_financieros_2024.xlsx   | existe: True
atm                       -> atm_x_10000_adultos_2024.xlsx            | existe: True
internet                  -> internet_hogares_2024.xlsx               | existe: True


## 4. Funciones auxiliares

Estas funciones ayudan a cargar archivos, limpiar encabezados y estandarizar nombres de departamentos.

In [4]:
def normalizar_columnas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace('á', 'a', regex=False)
        .str.replace('é', 'e', regex=False)
        .str.replace('í', 'i', regex=False)
        .str.replace('ó', 'o', regex=False)
        .str.replace('ú', 'u', regex=False)
        .str.replace('ñ', 'n', regex=False)
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )
    return df


def estandarizar_departamento(serie: pd.Series) -> pd.Series:
    s = serie.astype(str).str.strip().str.upper()
    s = (s
         .str.replace('Á', 'A', regex=False)
         .str.replace('É', 'E', regex=False)
         .str.replace('Í', 'I', regex=False)
         .str.replace('Ó', 'O', regex=False)
         .str.replace('Ú', 'U', regex=False))
    reemplazos = {
        'BOGOTA D.C.': 'BOGOTA D.C.',
        'BOGOTA, D.C.': 'BOGOTA D.C.',
        'SANTAFE DE BOGOTA D.C': 'BOGOTA D.C.',
        'SANTAFE DE BOGOTA D.C.': 'BOGOTA D.C.',
        'ARCHIPIELAGO DE SAN ANDRES PROVIDENCIA Y SANTA CATALINA': 'SAN ANDRES Y PROVIDENCIA',
        'ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA': 'SAN ANDRES Y PROVIDENCIA',
        'SAN ANDRES Y PROVIDENCIA': 'SAN ANDRES Y PROVIDENCIA',
        'NORTE DE SANTANDER': 'NORTE DE SANTANDER',
        'VALLE DEL CAUCA': 'VALLE DEL CAUCA',
        'LA GUAJIRA': 'LA GUAJIRA',
    }
    return s.replace(reemplazos)


def cargar_excel(ruta: Path, sheet_name=0) -> pd.DataFrame:
    df = pd.read_excel(ruta, sheet_name=sheet_name)
    df = normalizar_columnas(df)
    return df


def buscar_columna_departamento(df: pd.DataFrame) -> str:
    candidatas = ['departamento', 'nombre_dpt', 'dpto', 'departamentos']
    for c in candidatas:
        if c in df.columns:
            return c
    raise ValueError(f'No se encontró una columna de departamento en: {list(df.columns)}')

## 5. Carga inicial de datasets

In [5]:
dfs = {}

for nombre, ruta in FILES.items():
    dfs[nombre] = cargar_excel(ruta)
    print('=' * 80)
    print(f'DATASET: {nombre}')
    print('Forma:', dfs[nombre].shape)
    print('Columnas:', list(dfs[nombre].columns))
    display(dfs[nombre].head())

DATASET: pobreza
Forma: (24, 2)
Columnas: ['departamento', 'pobreza_2024']


,departamento,pobreza_2024
0,Antioquia,24.70
1,Atlántico,31.60
2,Bogotá D.C.,19.60
3,Bolívar,48.00
4,Boyacá,30.90


DATASET: microcredito
Forma: (33, 2)
Columnas: ['departamento', 'acceso_microcredito_2024']


,departamento,acceso_microcredito_2024
0,Huila,13.40
1,Putumayo,13.40
2,Nariño,12.70
3,Boyacá,12.30
4,Caquetá,12.20


DATASET: productos_financieros
Forma: (33, 2)
Columnas: ['departamento', 'acceso_productos_financieros_2024']


,departamento,acceso_productos_financieros_2024
0,Antioquia,126.30
1,Bogotá D.C.,118.80
2,Risaralda,102.50
3,Valle del Cauca,99.10
4,Huila,98.00


DATASET: atm
Forma: (33, 2)
Columnas: ['departamento', 'atm_x_10000_adultos_2024']


,departamento,atm_x_10000_adultos_2024
0,San Andrés y Providencia,8.80
1,Bogotá D.C.,7.50
2,Santander,4.90
3,Risaralda,4.90
4,Valle del Cauca,4.80


DATASET: internet
Forma: (33, 2)
Columnas: ['departamento', 'internet_hogares_2024']


,departamento,internet_hogares_2024
0,Bogotá D.C.,82.70
1,Meta,77.00
2,Tolima,76.50
3,Valle del Cauca,73.40
4,Risaralda,73.00


## 6. Estandarización mínima de la columna de departamento

Aquí se crea una columna común llamada `departamento` en todos los datasets.

In [6]:
for nombre, df in dfs.items():
    col_dep = buscar_columna_departamento(df)
    dfs[nombre] = df.copy()
    dfs[nombre]['departamento'] = estandarizar_departamento(dfs[nombre][col_dep])
    print(f'{nombre:25s} -> columna original: {col_dep}')

pobreza                   -> columna original: departamento
microcredito              -> columna original: departamento
productos_financieros     -> columna original: departamento
atm                       -> columna original: departamento
internet                  -> columna original: departamento


## 7. Revisión de departamentos por dataset

In [7]:
departamentos_por_fuente = {}

for nombre, df in dfs.items():
    deps = sorted(df['departamento'].dropna().unique().tolist())
    departamentos_por_fuente[nombre] = set(deps)
    print('=' * 80)
    print(f'{nombre.upper()} - cantidad de departamentos únicos: {len(deps)}')
    print(deps)

POBREZA - cantidad de departamentos únicos: 24
['ANTIOQUIA', 'ATLANTICO', 'BOGOTA D.C.', 'BOLIVAR', 'BOYACA', 'CALDAS', 'CAQUETA', 'CAUCA', 'CESAR', 'CHOCO', 'CORDOBA', 'CUNDINAMARCA', 'HUILA', 'LA GUAJIRA', 'MAGDALENA', 'META', 'NARIÑO', 'NORTE DE SANTANDER', 'QUINDIO', 'RISARALDA', 'SANTANDER', 'SUCRE', 'TOLIMA', 'VALLE DEL CAUCA']
MICROCREDITO - cantidad de departamentos únicos: 33
['AMAZONAS', 'ANTIOQUIA', 'ARAUCA', 'ATLANTICO', 'BOGOTA D.C.', 'BOLIVAR', 'BOYACA', 'CALDAS', 'CAQUETA', 'CASANARE', 'CAUCA', 'CESAR', 'CHOCO', 'CORDOBA', 'CUNDINAMARCA', 'GUAINIA', 'GUAVIARE', 'HUILA', 'LA GUAJIRA', 'MAGDALENA', 'META', 'NARIÑO', 'NORTE DE SANTANDER', 'PUTUMAYO', 'QUINDIO', 'RISARALDA', 'SAI', 'SANTANDER', 'SUCRE', 'TOLIMA', 'VALLE DEL CAUCA', 'VAUPES', 'VICHADA']
PRODUCTOS_FINANCIEROS - cantidad de departamentos únicos: 33
['AMAZONAS', 'ANTIOQUIA', 'ARAUCA', 'ATLANTICO', 'BOGOTA D.C.', 'BOLIVAR', 'BOYACA', 'CALDAS', 'CAQUETA', 'CASANARE', 'CAUCA', 'CESAR', 'CHOCO', 'CORDOBA', 'CUNDINAM

## 8. Comparación entre listas de departamentos

Esta revisión permite detectar nombres inconsistentes o territorios faltantes.

In [8]:
fuentes = list(departamentos_por_fuente.keys())
base = fuentes[0]
base_set = departamentos_por_fuente[base]

for fuente in fuentes[1:]:
    print('=' * 80)
    print(f'Comparación: {base} vs {fuente}')
    solo_base = sorted(base_set - departamentos_por_fuente[fuente])
    solo_fuente = sorted(departamentos_por_fuente[fuente] - base_set)
    print('Solo en base:', solo_base)
    print('Solo en fuente:', solo_fuente)

Comparación: pobreza vs microcredito
Solo en base: []
Solo en fuente: ['AMAZONAS', 'ARAUCA', 'CASANARE', 'GUAINIA', 'GUAVIARE', 'PUTUMAYO', 'SAI', 'VAUPES', 'VICHADA']
Comparación: pobreza vs productos_financieros
Solo en base: []
Solo en fuente: ['AMAZONAS', 'ARAUCA', 'CASANARE', 'GUAINIA', 'GUAVIARE', 'PUTUMAYO', 'SAN ANDRES Y PROVIDENCIA (SAI)', 'VAUPES', 'VICHADA']
Comparación: pobreza vs atm
Solo en base: []
Solo en fuente: ['AMAZONAS', 'ARAUCA', 'CASANARE', 'GUAINIA', 'GUAVIARE', 'PUTUMAYO', 'SAN ANDRES Y PROVIDENCIA', 'VAUPES', 'VICHADA']
Comparación: pobreza vs internet
Solo en base: []
Solo en fuente: ['AMAZONAS', 'ARAUCA', 'CASANARE', 'GUAINIA', 'GUAVIARE', 'PUTUMAYO', 'SAN ANDRES', 'VAUPES', 'VICHADA']


## 9. Selección preliminar de columnas de valor




In [9]:
COLUMNAS_VALOR = {
    'pobreza': 'pobreza_2024',
    'microcredito': 'acceso_microcredito_2024',
    'productos_financieros': 'acceso_productos_financieros_2024',
    'atm': 'atm_x_10000_adultos_2024',
    'internet': 'internet_hogares_2024',
}

for nombre, col in COLUMNAS_VALOR.items():
    print(f'{nombre:25s} -> columna esperada: {col} | existe: {col in dfs[nombre].columns}')

pobreza                   -> columna esperada: pobreza_2024 | existe: True
microcredito              -> columna esperada: acceso_microcredito_2024 | existe: True
productos_financieros     -> columna esperada: acceso_productos_financieros_2024 | existe: True
atm                       -> columna esperada: atm_x_10000_adultos_2024 | existe: True
internet                  -> columna esperada: internet_hogares_2024 | existe: True


## 10. Construcción de una vista preliminar estandarizada

cada archivo es reducido a:
- `departamento`
- la variable principal correspondiente

In [10]:
vistas_limpias = {}

for nombre, df in dfs.items():
    col_valor = COLUMNAS_VALOR[nombre]
    temp = df[['departamento', col_valor]].copy()
    temp = temp.dropna(subset=['departamento'])
    temp = temp.drop_duplicates(subset=['departamento'])
    vistas_limpias[nombre] = temp
    print('=' * 80)
    print(nombre)
    print(temp.shape)
    display(temp.head())

pobreza
(24, 2)


,departamento,pobreza_2024
0,ANTIOQUIA,24.70
1,ATLANTICO,31.60
2,BOGOTA D.C.,19.60
3,BOLIVAR,48.00
4,BOYACA,30.90


microcredito
(33, 2)


,departamento,acceso_microcredito_2024
0,HUILA,13.40
1,PUTUMAYO,13.40
2,NARIÑO,12.70
3,BOYACA,12.30
4,CAQUETA,12.20


productos_financieros
(33, 2)


,departamento,acceso_productos_financieros_2024
0,ANTIOQUIA,126.30
1,BOGOTA D.C.,118.80
2,RISARALDA,102.50
3,VALLE DEL CAUCA,99.10
4,HUILA,98.00


atm
(33, 2)


,departamento,atm_x_10000_adultos_2024
0,SAN ANDRES Y PROVIDENCIA,8.80
1,BOGOTA D.C.,7.50
2,SANTANDER,4.90
3,RISARALDA,4.90
4,VALLE DEL CAUCA,4.80


internet
(33, 2)


,departamento,internet_hogares_2024
0,BOGOTA D.C.,82.70
1,META,77.00
2,TOLIMA,76.50
3,VALLE DEL CAUCA,73.40
4,RISARALDA,73.00


## 11. Resumen de calidad básica

Aquí se revisan valores nulos y duplicados por dataset.

In [11]:
resumen_calidad = []

for nombre, df in vistas_limpias.items():
    col_valor = [c for c in df.columns if c != 'departamento'][0]
    resumen_calidad.append({
        'dataset': nombre,
        'filas': len(df),
        'departamentos_unicos': df['departamento'].nunique(),
        'nulos_valor': int(df[col_valor].isna().sum()),
        'duplicados_departamento': int(df['departamento'].duplicated().sum()),
    })

resumen_calidad = pd.DataFrame(resumen_calidad)
display(resumen_calidad)

,dataset,filas,departamentos_unicos,nulos_valor,duplicados_departamento
0,pobreza,24,24,0,0
1,microcredito,33,33,0,0
2,productos_financieros,33,33,0,0
3,atm,33,33,0,0
4,internet,33,33,0,0
